# 03 — Dataset de Treino (Neural Collaborative Filtering)

**Tech Challenge Fase 02 — Sistema de Recomendação (Instacart)**

Prepara os exemplos de treino para o modelo NCF (framing escolhido: embeddings
de usuário e produto + amostragem de negativos). Cada exemplo é um par
(usuário, produto) com um rótulo:
- **1** = o usuário interagiu (comprou no histórico).
- **0** = negativo amostrado (produto que o usuário não comprou).

**Saídas (em `data/processed/`):**
- `train_positives.parquet` — pares positivos `(user_idx, product_idx, label=1)`.

A amostragem de negativos é entregue como **função reutilizável**, para ser
chamada **dinamicamente no treino** (notebook 05) — re-amostrando a cada época.

## 0. Setup

In [1]:
import random
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

pd.set_option('display.max_columns', 50)


def find_project_root(marker: str = 'pyproject.toml') -> Path:
    '''Sobe na arvore de diretorios ate encontrar o marcador do projeto.'''
    current = Path.cwd()
    for parent in [current, *current.parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f'Marcador {marker} nao encontrado a partir de {current}')


PROJECT_ROOT = find_project_root()
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
print('Processed:', PROCESSED_DIR)

Processed: c:\Users\erick\projetos\mlp-market-recommender-system\data\processed


## 1. Carregar as interações

Partimos do artefato do notebook 02 (`interactions.parquet`). Dele derivamos o
número de usuários e de produtos — que definem o **tamanho das tabelas de
embedding** (uma linha por índice).

In [2]:
interactions = pd.read_parquet(PROCESSED_DIR / 'interactions.parquet')

n_users = int(interactions['user_idx'].max()) + 1
n_items = int(interactions['product_idx'].max()) + 1
print('Interações:', f'{len(interactions):,}')
print('n_users   :', f'{n_users:,}')
print('n_items   :', f'{n_items:,}')
interactions.head()

Interações: 13,307,953
n_users   : 206,209
n_items   : 49,677


,user_id,product_id,n_orders,n_reorders,user_idx,product_idx
0,1,196,10,9,0,195
1,1,10258,9,8,0,10254
2,1,10326,1,0,0,10322
3,1,12427,10,9,0,12423
4,1,13032,3,2,0,13028


## 2. Pares positivos

Todo par (usuário, produto) observado no histórico é um **exemplo positivo**
(rótulo 1). Para o NCF padrão usamos o sinal binário (interagiu ou não); a força
da interação (`n_orders`) fica disponível para uma extensão futura.

In [3]:
def build_positives(interactions: pd.DataFrame) -> pd.DataFrame:
    '''Monta os pares positivos (interagiu = 1) para o treino do NCF.'''
    positives = interactions[['user_idx', 'product_idx']].copy()
    positives['label'] = np.int8(1)
    return positives


positives = build_positives(interactions)
print('Positivos:', f'{len(positives):,}')
positives.head()

Positivos: 13,307,953


,user_idx,product_idx,label
0,0,195,1
1,0,10254,1
2,0,10322,1
3,0,12423,1
4,0,13028,1


## 3. Amostragem de negativos

Para cada positivo, sorteamos `n_neg` produtos aleatórios que o usuário **não**
comprou. Um negativo que por acaso seja um positivo do próprio usuário é removido
(colisão) — raro, dada a esparsidade de ~99,87%.

A função é vetorizada (rápida) e usa uma *seed* para ser reprodutível. No treino
(notebook 05) ela será chamada **a cada época**, gerando negativos novos.

In [4]:
def sample_negatives(
    positives: pd.DataFrame, n_items: int, n_neg: int, seed: int
) -> pd.DataFrame:
    '''Amostra n_neg negativos por positivo (produtos aleatorios nao comprados).'''
    rng = np.random.default_rng(seed)
    users = np.repeat(positives['user_idx'].to_numpy(), n_neg)
    items = rng.integers(0, n_items, size=users.shape[0], dtype='int64')
    pos_keys = (
        positives['user_idx'].to_numpy().astype('int64') * n_items
        + positives['product_idx'].to_numpy()
    )
    neg_keys = users.astype('int64') * n_items + items
    keep = ~np.isin(neg_keys, np.sort(pos_keys))
    return pd.DataFrame(
        {
            'user_idx': users[keep].astype('int32'),
            'product_idx': items[keep].astype('int32'),
            'label': np.zeros(int(keep.sum()), dtype='int8'),
        }
    )

### Demonstração em pequena escala

Validamos a função numa amostra (100 mil positivos, 2 negativos cada) — confirma
o tamanho gerado e quantas colisões foram removidas.

In [5]:
demo_pos = positives.head(100_000)
demo_neg = sample_negatives(demo_pos, n_items, n_neg=2, seed=SEED)
print('Positivos (demo) :', f'{len(demo_pos):,}')
print('Negativos (demo) :', f'{len(demo_neg):,}')
print('Colisões removidas:', f'{2 * len(demo_pos) - len(demo_neg):,}')
demo_neg.head()

Positivos (demo) : 100,000
Negativos (demo) : 199,561
Colisões removidas: 439


,user_idx,product_idx,label
0,0,4433,0
1,0,38447,0
2,0,32517,0
3,0,21802,0
4,0,21510,0


## 4. Salvar os positivos

Salvamos apenas os **positivos** (os negativos serão amostrados no treino). Esse
é o insumo de treino canônico para o notebook 05.

In [6]:
def save_parquet(df: pd.DataFrame, path: Path) -> None:
    '''Salva um DataFrame em parquet e informa o resultado.'''
    df.to_parquet(path, index=False)
    print(f'  saved {path.name}  ({len(df):,} linhas)')


save_parquet(positives, PROCESSED_DIR / 'train_positives.parquet')

  saved train_positives.parquet  (13,307,953 linhas)


## 5. Resumo

- **`train_positives.parquet`** — pares positivos (user_idx, product_idx, label=1).
- **`sample_negatives(...)`** — função para amostrar negativos, a ser chamada
  dinamicamente no treino (re-amostragem por época). Razão recomendada: **4:1**.

**Nota de compute:** como o PyTorch está em build CPU, no notebook 05
provavelmente treinaremos num **subconjunto de usuários** para o tempo de treino
ser viável — decisão a tomar lá.

**Backlog de refatoração (para `src/` na fase final):** `build_positives` e
`sample_negatives` → `src/data/` ou `src/features/`.

**Próximo notebook (`04`):** baselines (popularidade) e as **métricas de ranking**
(Precision@K, Recall@K, NDCG@K, MAP@K) que servirão para comparar os modelos.